In [14]:
from pathlib import Path
import re
import time
import warnings

import fastf1
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb

from fastf1.ergast import Ergast
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

RANDOM_STATE = 42

# Lap timing is mainly available from 2018 onward.
# Fewer seasons download faster. Add older seasons after this version works.
START_YEAR = 2022
END_YEAR = 2025
TEST_YEAR = 2025

DATA_FOLDER = Path("session_model_data")
SESSION_FOLDER = DATA_FOLDER / "processed_sessions"
CACHE_FOLDER = DATA_FOLDER / "fastf1_cache"
ARTIFACT_FOLDER = Path("artifacts")

for folder in [
    DATA_FOLDER,
    SESSION_FOLDER,
    CACHE_FOLDER,
    ARTIFACT_FOLDER,
]:
    folder.mkdir(parents=True, exist_ok=True)

RACE_RESULTS_FILE = DATA_FOLDER / "race_results.csv"
WEEKEND_FEATURE_FILE = DATA_FOLDER / "weekend_lap_features.csv"
MODEL_FILE = ARTIFACT_FOLDER / "f1_session_lap_winner_model.joblib"

fastf1.Cache.enable_cache(str(CACHE_FOLDER))

# Change to True only when you intentionally want to rebuild saved files.
DOWNLOAD_RESULTS_AGAIN = False
REBUILD_SESSION_FILES = False


In [28]:
def get_value(row, possible_columns, default=np.nan):
    for column in possible_columns:
        if column in row.index and pd.notna(row[column]):
            return row[column]
    return default


In [27]:
def download_race_results(start_year, end_year):
    ergast = Ergast(result_type="pandas", auto_cast=True, limit=1000)
    rows = []

    for year in range(start_year, end_year + 1):
        print("Race results:", year)
        response = ergast.get_race_results(season=year, limit=1000)
        race_details = response.description.reset_index(drop=True)

        for race_index, result_table in enumerate(response.content):
            race = race_details.iloc[race_index]
            results = pd.DataFrame(result_table)
            race_round = int(get_value(race, ["round"], race_index + 1))

            for _, result in results.iterrows():
                finish = pd.to_numeric(
                    get_value(result, ["position", "resultPosition"]),
                    errors="coerce",
                )
                if pd.isna(finish):
                    continue

                rows.append(
                    {
                        "season": year,
                        "round": race_round,
                        "race_name": get_value(race, ["raceName"]),
                        "race_date": pd.to_datetime(
                            get_value(race, ["raceDate", "date"], pd.NaT)
                        ),
                        "circuit": get_value(
                            race,
                            ["circuitName", "circuitId", "locality"],
                        ),
                        "driver": get_value(
                            result, ["driverCode", "driverId"], "Unknown"
                        ),
                        "team": get_value(
                            result,
                            ["constructorName", "constructorId"],
                            "Unknown",
                        ),
                        "grid_position": pd.to_numeric(
                            get_value(
                                result, ["grid", "gridPosition"]
                            ),
                            errors="coerce",
                        ),
                        "finish_position": finish,
                        "winner": int(finish == 1),
                    }
                )

    return pd.DataFrame(rows)

In [29]:
if RACE_RESULTS_FILE.exists() and not DOWNLOAD_RESULTS_AGAIN:
    race_results = pd.read_csv(
        RACE_RESULTS_FILE,
        parse_dates=["race_date"],
    )
else:
    race_results = download_race_results(START_YEAR, END_YEAR)
    race_results.to_csv(RACE_RESULTS_FILE, index=False)

In [30]:
race_results.loc[
    race_results["grid_position"] == 0,
    "grid_position",
] = np.nan

print("Race-result rows:", len(race_results))
print(
    "Grand Prix races:",
    race_results.groupby(["season", "round"]).ngroups,
)
race_results.head()

Race-result rows: 400
Grand Prix races: 21


,season,round,race_name,race_date,circuit,driver,team,grid_position,finish_position,winner
0,2022,1,Bahrain Grand Prix,2022-03-20,Bahrain International Circuit,LEC,Ferrari,1.0,1,1
1,2022,1,Bahrain Grand Prix,2022-03-20,Bahrain International Circuit,SAI,Ferrari,3.0,2,0
2,2022,1,Bahrain Grand Prix,2022-03-20,Bahrain International Circuit,HAM,Mercedes,5.0,3,0
3,2022,1,Bahrain Grand Prix,2022-03-20,Bahrain International Circuit,RUS,Mercedes,9.0,4,0
4,2022,1,Bahrain Grand Prix,2022-03-20,Bahrain International Circuit,MAG,Haas F1 Team,7.0,5,0


In [ ]:
SESSION_NAME_MAP = {
    "practice 1": "fp1",
    "fp1": "fp1",
    "practice 2": "fp2",
    "fp2": "fp2",
    "practice 3": "fp3",
    "fp3": "fp3",
    "qualifying": "quali",
    "sprint shootout": "sprint_quali",
    "sprint qualifying": "sprint_quali",
    "sprint": "sprint",
}

In [31]:
def simple_session_name(session_name):
    return SESSION_NAME_MAP.get(str(session_name).strip().lower())


def seconds(series):
    return pd.to_timedelta(series, errors="coerce").dt.total_seconds()


def safe_file_name(text):
    return re.sub(r"[^A-Za-z0-9_-]+", "_", str(text))


def clean_session_laps(laps):
    laps = pd.DataFrame(laps).copy()
    laps["lap_seconds"] = seconds(laps["LapTime"])

    # Remove laps without a time and pit entry/exit laps.
    laps = laps[laps["lap_seconds"].notna()].copy()
    laps = laps[
        laps["PitInTime"].isna() & laps["PitOutTime"].isna()
    ].copy()

    if "Deleted" in laps.columns:
        laps = laps[laps["Deleted"].fillna(False) == False].copy()

    if "FastF1Generated" in laps.columns:
        laps = laps[
            laps["FastF1Generated"].fillna(False) == False
        ].copy()

    # Prefer laps that pass FastF1's timing-accuracy checks.
    # Use this filter only when it leaves at least one lap per driver.
    if "IsAccurate" in laps.columns:
        accurate_laps = laps[laps["IsAccurate"].fillna(False)].copy()
        if (
            not accurate_laps.empty
            and accurate_laps["Driver"].nunique()
            == laps["Driver"].nunique()
        ):
            laps = accurate_laps

    # Convert sector times.
    for sector in ["Sector1Time", "Sector2Time", "Sector3Time"]:
        if sector in laps.columns:
            laps[sector + "_seconds"] = seconds(laps[sector])

    # Remove extreme timing values separately for each driver.
    driver_median = laps.groupby("Driver")["lap_seconds"].transform(
        "median"
    )
    laps = laps[
        laps["lap_seconds"].between(
            driver_median * 0.80,
            driver_median * 1.20,
        )
    ].copy()

    return laps

In [32]:
def summarize_driver_laps(laps, session_code, session_results=None):
    rows = []

    for driver, driver_laps in laps.groupby("Driver"):
        lap_times = driver_laps["lap_seconds"].sort_values()

        row = {
            "driver": driver,
            f"{session_code}_lap_count": len(lap_times),
            f"{session_code}_best_lap": lap_times.min(),
            f"{session_code}_median_lap": lap_times.median(),
            f"{session_code}_top3_average": (
                lap_times.head(3).mean()
            ),
            f"{session_code}_lap_std": lap_times.std(),
        }

        for number in [1, 2, 3]:
            sector_column = f"Sector{number}Time_seconds"
            row[f"{session_code}_best_sector_{number}"] = (
                driver_laps[sector_column].min()
                if sector_column in driver_laps.columns
                else np.nan
            )

        rows.append(row)

    summary = pd.DataFrame(rows)
    if summary.empty:
        return summary

    # Convert absolute lap and sector times to gaps from session best.
    pace_columns = [
        "best_lap",
        "median_lap",
        "top3_average",
        "best_sector_1",
        "best_sector_2",
        "best_sector_3",
    ]

    for ending in pace_columns:
        column = f"{session_code}_{ending}"
        best_value = summary[column].min()
        summary[column + "_gap"] = summary[column] - best_value

    # Add Sprint finishing position because Sprint happens before the GP.
    if session_code == "sprint" and session_results is not None:
        results = pd.DataFrame(session_results).copy()
        if not results.empty:
            sprint_positions = results[
                ["Abbreviation", "Position"]
            ].rename(
                columns={
                    "Abbreviation": "driver",
                    "Position": "sprint_finish_position",
                }
            )
            summary = summary.merge(
                sprint_positions,
                on="driver",
                how="left",
            )

    return summary

In [33]:
def process_one_session(year, race_round, session_number, session_name):
    session_code = simple_session_name(session_name)
    if session_code is None:
        return None

    file_name = (
        f"{year}_{race_round:02d}_{session_number}_"
        f"{safe_file_name(session_code)}.csv"
    )
    saved_file = SESSION_FOLDER / file_name

    if saved_file.exists() and not REBUILD_SESSION_FILES:
        return pd.read_csv(saved_file)

    print(
        f"  Loading {year} round {race_round}: {session_name}"
    )

    session = fastf1.get_session(year, race_round, session_number)
    session.load(
        laps=True,
        telemetry=False,
        weather=False,
        messages=False,
    )

    clean_laps = clean_session_laps(session.laps)
    summary = summarize_driver_laps(
        clean_laps,
        session_code,
        session.results,
    )

    if summary.empty:
        return None

    summary.insert(0, "season", year)
    summary.insert(1, "round", race_round)
    summary.to_csv(saved_file, index=False)

    # A small pause is polite to the data service.
    time.sleep(1)
    return summary


In [34]:
def collect_all_session_features(start_year, end_year):
    all_sessions = []

    for year in range(start_year, end_year + 1):
        print(f"\nSeason {year}")
        schedule = fastf1.get_event_schedule(
            year,
            include_testing=False,
        )

        for _, event in schedule.iterrows():
            race_round = int(event["RoundNumber"])

            # Future or unavailable races do not have a result label.
            race_exists = (
                (race_results["season"] == year)
                & (race_results["round"] == race_round)
            ).any()
            if not race_exists:
                continue

            print(f"Round {race_round}: {event['EventName']}")

            for session_number in range(1, 6):
                name_column = f"Session{session_number}"
                session_name = event.get(name_column)

                if pd.isna(session_name):
                    continue

                # Race is deliberately excluded.
                if simple_session_name(session_name) is None:
                    continue

                try:
                    summary = process_one_session(
                        year,
                        race_round,
                        session_number,
                        session_name,
                    )
                    if summary is not None:
                        all_sessions.append(summary)

                except Exception as error:
                    print("    Could not load:", error)
                    if (
                        "500 calls/h" in str(error)
                        or "RateLimit" in type(error).__name__
                    ):
                        print(
                            "FastF1 hourly limit reached. Stop the cell, "
                            "wait for the limit to reset, then rerun. "
                            "Completed sessions are already saved."
                        )
                        raise

    return all_sessions

In [35]:
session_tables = collect_all_session_features(
    START_YEAR,
    END_YEAR,
)

print("\nProcessed session tables:", len(session_tables))


Season 2022
Round 1: Bahrain Grand Prix
Round 2: Saudi Arabian Grand Prix
Round 3: Australian Grand Prix
Round 4: Emilia Romagna Grand Prix
Round 5: Miami Grand Prix

Season 2023
Round 1: Bahrain Grand Prix
Round 2: Saudi Arabian Grand Prix
Round 3: Australian Grand Prix
Round 4: Azerbaijan Grand Prix
Round 5: Miami Grand Prix

Season 2024
Round 1: Bahrain Grand Prix
Round 2: Saudi Arabian Grand Prix
Round 3: Australian Grand Prix
Round 4: Japanese Grand Prix
Round 5: Chinese Grand Prix
Round 6: Miami Grand Prix

Season 2025
Round 1: Australian Grand Prix
Round 2: Chinese Grand Prix
Round 3: Japanese Grand Prix
Round 4: Bahrain Grand Prix
Round 5: Saudi Arabian Grand Prix

Processed session tables: 84


In [37]:
if not session_tables:
    # This also allows rebuilding the combined file from saved session CSVs.
    session_tables = [
        pd.read_csv(file)
        for file in SESSION_FOLDER.glob("*.csv")
    ]

if not session_tables:
    raise ValueError("No session data was collected.")

In [40]:
keys = ["season", "round", "driver"]

# Every saved file represents one session from one race.
# Files from different races can have the same columns, such as FP1 columns.
# Therefore, stack the files vertically instead of repeatedly merging them.
all_session_rows = pd.concat(
    session_tables,
    ignore_index=True,
    sort=False,
)

# Combine FP1, FP2, FP3, Qualifying and Sprint values into one row for each
# driver in each race. `first()` selects the available non-missing value.
weekend_features = (
    all_session_rows
    .groupby(keys, as_index=False)
    .first()
)

weekend_features.to_csv(WEEKEND_FEATURE_FILE, index=False)

print("Session-file rows:", len(all_session_rows))
print("Weekend feature rows:", len(weekend_features))
print("Weekend feature columns:", len(weekend_features.columns))
weekend_features.head()


Session-file rows: 1636
Weekend feature rows: 429
Weekend feature columns: 88


,season,round,driver,fp1_lap_count,fp1_best_lap,fp1_median_lap,fp1_top3_average,fp1_lap_std,fp1_best_sector_1,fp1_best_sector_2,fp1_best_sector_3,fp1_best_lap_gap,fp1_median_lap_gap,fp1_top3_average_gap,fp1_best_sector_1_gap,fp1_best_sector_2_gap,fp1_best_sector_3_gap,fp2_lap_count,fp2_best_lap,fp2_median_lap,fp2_top3_average,fp2_lap_std,fp2_best_sector_1,fp2_best_sector_2,fp2_best_sector_3,fp2_best_lap_gap,fp2_median_lap_gap,fp2_top3_average_gap,fp2_best_sector_1_gap,fp2_best_sector_2_gap,fp2_best_sector_3_gap,fp3_lap_count,fp3_best_lap,fp3_median_lap,fp3_top3_average,fp3_lap_std,fp3_best_sector_1,fp3_best_sector_2,fp3_best_sector_3,fp3_best_lap_gap,fp3_median_lap_gap,fp3_top3_average_gap,fp3_best_sector_1_gap,fp3_best_sector_2_gap,fp3_best_sector_3_gap,quali_lap_count,quali_best_lap,quali_median_lap,quali_top3_average,quali_lap_std,quali_best_sector_1,quali_best_sector_2,quali_best_sector_3,quali_best_lap_gap,quali_median_lap_gap,quali_top3_average_gap,quali_best_sector_1_gap,quali_best_sector_2_gap,quali_best_sector_3_gap,sprint_lap_count,sprint_best_lap,sprint_median_lap,sprint_top3_average,sprint_lap_std,sprint_best_sector_1,sprint_best_sector_2,sprint_best_sector_3,sprint_best_lap_gap,sprint_median_lap_gap,sprint_top3_average_gap,sprint_best_sector_1_gap,sprint_best_sector_2_gap,sprint_best_sector_3_gap,sprint_finish_position,sprint_quali_lap_count,sprint_quali_best_lap,sprint_quali_median_lap,sprint_quali_top3_average,sprint_quali_lap_std,sprint_quali_best_sector_1,sprint_quali_best_sector_2,sprint_quali_best_sector_3,sprint_quali_best_lap_gap,sprint_quali_median_lap_gap,sprint_quali_top3_average_gap,sprint_quali_best_sector_1_gap,sprint_quali_best_sector_2_gap,sprint_quali_best_sector_3_gap
0,2022,1,ALB,7.0,95.923,98.9200,96.851000,9.389774,30.219,41.475,24.015,1.730,3.6610,1.917000,0.148,0.983,0.410,12.0,94.735,100.0200,95.406000,6.025374,30.128,40.667,23.807,2.799,6.5610,2.937000,0.752,1.222,0.754,4.0,94.868,95.5790,95.342000,0.526032,30.189,40.906,23.773,2.324,1.7820,1.955000,0.615,1.334,0.680,3.0,92.664,92.726,92.822000,0.222144,29.564,39.785,23.203,2.106,1.969,2.095000,0.594,1.083,0.496,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022,1,ALO,6.0,95.000,95.2590,95.142000,6.580376,30.292,40.826,23.822,0.807,0.0000,0.208000,0.221,0.334,0.217,14.0,92.877,98.2450,93.548333,4.943994,29.800,39.682,23.395,0.941,4.7860,1.079333,0.424,0.237,0.342,8.0,94.628,101.1255,95.879000,9.089134,30.202,40.870,23.556,2.084,7.3285,2.492000,0.628,1.298,0.463,4.0,91.621,91.999,91.873000,0.336593,29.301,39.305,23.010,1.063,1.242,1.146000,0.331,0.603,0.303,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2022,1,BOT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,92.951,97.8080,93.157000,5.657683,29.662,39.800,23.324,1.015,4.3490,0.688000,0.286,0.355,0.271,7.0,93.733,94.6170,93.827667,1.788776,29.918,40.138,23.348,1.189,0.8200,0.440667,0.344,0.566,0.255,5.0,91.560,91.817,91.698000,0.342295,29.203,39.296,22.861,1.002,1.060,0.971000,0.233,0.594,0.154,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2022,1,GAS,14.0,94.193,97.6625,95.427667,7.341504,30.076,40.492,23.625,0.000,2.4035,0.493667,0.005,0.000,0.020,12.0,93.621,98.2965,94.111667,2.271603,29.655,40.323,23.590,1.685,4.8375,1.642667,0.279,0.878,0.537,9.0,94.176,96.8050,94.890000,14.915578,29.978,40.544,23.624,1.632,3.0080,1.503000,0.404,0.972,0.531,7.0,91.635,92.338,92.015333,0.564460,29.253,39.335,23.018,1.077,1.581,1.288333,0.283,0.633,0.311,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2022,1,HAM,4.0,94.943,97.2475,96.479333,3.632460,30.512,40.613,23.818,0.750,1.9885,1.545333,0.441,0.121,0.213,16.0,93.144,98.8205,95.183000,6.490954,30.115,39.694,23.335,1.208,5.3615,2.714000,0.739,0.249,0.282,4.0

In [39]:
if not session_tables:
    raise ValueError("No session data was collected.")

keys = ["season", "round", "driver"]

# Every saved file represents one session from one race.
# Files from different races can have the same columns, such as FP1 columns.
# Therefore, stack the files vertically instead of repeatedly merging them.
all_session_rows = pd.concat(
    session_tables,
    ignore_index=True,
    sort=False,
)

# Combine FP1, FP2, FP3, Qualifying and Sprint values into one row for each
# driver in each race. `first()` selects the available non-missing value.
weekend_features = (
    all_session_rows
    .groupby(keys, as_index=False)
    .first()
)

weekend_features.to_csv(WEEKEND_FEATURE_FILE, index=False)

print("Session-file rows:", len(all_session_rows))
print("Weekend feature rows:", len(weekend_features))
print("Weekend feature columns:", len(weekend_features.columns))
weekend_features.head()


Session-file rows: 1636
Weekend feature rows: 429
Weekend feature columns: 88


,season,round,driver,fp1_lap_count,fp1_best_lap,fp1_median_lap,fp1_top3_average,fp1_lap_std,fp1_best_sector_1,fp1_best_sector_2,fp1_best_sector_3,fp1_best_lap_gap,fp1_median_lap_gap,fp1_top3_average_gap,fp1_best_sector_1_gap,fp1_best_sector_2_gap,fp1_best_sector_3_gap,fp2_lap_count,fp2_best_lap,fp2_median_lap,fp2_top3_average,fp2_lap_std,fp2_best_sector_1,fp2_best_sector_2,fp2_best_sector_3,fp2_best_lap_gap,fp2_median_lap_gap,fp2_top3_average_gap,fp2_best_sector_1_gap,fp2_best_sector_2_gap,fp2_best_sector_3_gap,fp3_lap_count,fp3_best_lap,fp3_median_lap,fp3_top3_average,fp3_lap_std,fp3_best_sector_1,fp3_best_sector_2,fp3_best_sector_3,fp3_best_lap_gap,fp3_median_lap_gap,fp3_top3_average_gap,fp3_best_sector_1_gap,fp3_best_sector_2_gap,fp3_best_sector_3_gap,quali_lap_count,quali_best_lap,quali_median_lap,quali_top3_average,quali_lap_std,quali_best_sector_1,quali_best_sector_2,quali_best_sector_3,quali_best_lap_gap,quali_median_lap_gap,quali_top3_average_gap,quali_best_sector_1_gap,quali_best_sector_2_gap,quali_best_sector_3_gap,sprint_lap_count,sprint_best_lap,sprint_median_lap,sprint_top3_average,sprint_lap_std,sprint_best_sector_1,sprint_best_sector_2,sprint_best_sector_3,sprint_best_lap_gap,sprint_median_lap_gap,sprint_top3_average_gap,sprint_best_sector_1_gap,sprint_best_sector_2_gap,sprint_best_sector_3_gap,sprint_finish_position,sprint_quali_lap_count,sprint_quali_best_lap,sprint_quali_median_lap,sprint_quali_top3_average,sprint_quali_lap_std,sprint_quali_best_sector_1,sprint_quali_best_sector_2,sprint_quali_best_sector_3,sprint_quali_best_lap_gap,sprint_quali_median_lap_gap,sprint_quali_top3_average_gap,sprint_quali_best_sector_1_gap,sprint_quali_best_sector_2_gap,sprint_quali_best_sector_3_gap
0,2022,1,ALB,7.0,95.923,98.9200,96.851000,9.389774,30.219,41.475,24.015,1.730,3.6610,1.917000,0.148,0.983,0.410,12.0,94.735,100.0200,95.406000,6.025374,30.128,40.667,23.807,2.799,6.5610,2.937000,0.752,1.222,0.754,4.0,94.868,95.5790,95.342000,0.526032,30.189,40.906,23.773,2.324,1.7820,1.955000,0.615,1.334,0.680,3.0,92.664,92.726,92.822000,0.222144,29.564,39.785,23.203,2.106,1.969,2.095000,0.594,1.083,0.496,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022,1,ALO,6.0,95.000,95.2590,95.142000,6.580376,30.292,40.826,23.822,0.807,0.0000,0.208000,0.221,0.334,0.217,14.0,92.877,98.2450,93.548333,4.943994,29.800,39.682,23.395,0.941,4.7860,1.079333,0.424,0.237,0.342,8.0,94.628,101.1255,95.879000,9.089134,30.202,40.870,23.556,2.084,7.3285,2.492000,0.628,1.298,0.463,4.0,91.621,91.999,91.873000,0.336593,29.301,39.305,23.010,1.063,1.242,1.146000,0.331,0.603,0.303,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2022,1,BOT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,92.951,97.8080,93.157000,5.657683,29.662,39.800,23.324,1.015,4.3490,0.688000,0.286,0.355,0.271,7.0,93.733,94.6170,93.827667,1.788776,29.918,40.138,23.348,1.189,0.8200,0.440667,0.344,0.566,0.255,5.0,91.560,91.817,91.698000,0.342295,29.203,39.296,22.861,1.002,1.060,0.971000,0.233,0.594,0.154,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2022,1,GAS,14.0,94.193,97.6625,95.427667,7.341504,30.076,40.492,23.625,0.000,2.4035,0.493667,0.005,0.000,0.020,12.0,93.621,98.2965,94.111667,2.271603,29.655,40.323,23.590,1.685,4.8375,1.642667,0.279,0.878,0.537,9.0,94.176,96.8050,94.890000,14.915578,29.978,40.544,23.624,1.632,3.0080,1.503000,0.404,0.972,0.531,7.0,91.635,92.338,92.015333,0.564460,29.253,39.335,23.018,1.077,1.581,1.288333,0.283,0.633,0.311,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2022,1,HAM,4.0,94.943,97.2475,96.479333,3.632460,30.512,40.613,23.818,0.750,1.9885,1.545333,0.441,0.121,0.213,16.0,93.144,98.8205,95.183000,6.490954,30.115,39.694,23.335,1.208,5.3615,2.714000,0.739,0.249,0.282,4.0

In [42]:
model_data = race_results.merge(
    weekend_features,
    on=["season", "round", "driver"],
    how="inner",
)


In [43]:
# Remove columns that reveal the race result.
display_columns = [
    "season",
    "round",
    "race_name",
    "race_date",
    "circuit",
    "driver",
    "team",
    "grid_position",
    "finish_position",
    "winner",
]

# These are identifiers or result labels, not model inputs.
# Grid position is intentionally NOT excluded.
excluded_columns = {
    "season",
    "round",
    "race_name",
    "race_date",
    "circuit",
    "driver",
    "team",
    "finish_position",
    "winner",
}

In [ ]:
numeric_features = [
    column
    for column in model_data.columns
    if column not in excluded_columns
    and pd.api.types.is_numeric_dtype(model_data[column])
]

# Circuit identity is useful because pace characteristics differ by track.
category_features = ["circuit"]
feature_columns = category_features + numeric_features

In [ ]:
print("Races with session data:", model_data.groupby(["season", "round"]).ngroups)
print("Numeric session features:", len(numeric_features))
print("Driver name is a feature:", "driver" in feature_columns)
print("Team name is a feature:", "team" in feature_columns)
model_data[display_columns].head()

Races with session data: 21
Numeric session features: 86
Driver name is a feature: False
Team name is a feature: False


,season,round,race_name,race_date,circuit,driver,team,grid_position,finish_position,winner
0,2022,1,Bahrain Grand Prix,2022-03-20,Bahrain International Circuit,LEC,Ferrari,1.0,1,1
1,2022,1,Bahrain Grand Prix,2022-03-20,Bahrain International Circuit,SAI,Ferrari,3.0,2,0
2,2022,1,Bahrain Grand Prix,2022-03-20,Bahrain International Circuit,HAM,Mercedes,5.0,3,0
3,2022,1,Bahrain Grand Prix,2022-03-20,Bahrain International Circuit,RUS,Mercedes,9.0,4,0
4,2022,1,Bahrain Grand Prix,2022-03-20,Bahrain International Circuit,MAG,Haas F1 Team,7.0,5,0


In [ ]:
train_data = model_data[model_data["season"] < TEST_YEAR].copy()
test_data = model_data[model_data["season"] == TEST_YEAR].copy()

In [ ]:
if test_data.empty:
    TEST_YEAR = int(model_data["season"].max())
    train_data = model_data[
        model_data["season"] < TEST_YEAR
    ].copy()
    test_data = model_data[
        model_data["season"] == TEST_YEAR
    ].copy()


In [47]:
validation_year = int(train_data["season"].max())

X_train = train_data[feature_columns]
y_train = train_data["winner"]
X_test = test_data[feature_columns]
y_test = test_data["winner"]

In [48]:
print("Training seasons:", sorted(train_data["season"].unique()))
print("Validation season:", validation_year)
print("Unseen test season:", TEST_YEAR)
print("Training races:", train_data.groupby(["season", "round"]).ngroups)
print("Testing races:", test_data.groupby(["season", "round"]).ngroups)


Training seasons: [np.int64(2022), np.int64(2023), np.int64(2024)]
Validation season: 2024
Unseen test season: 2025
Training races: 16
Testing races: 5


In [49]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "fill_missing",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
            ),
        ),
    ]
)

category_pipeline = Pipeline(
    steps=[
        (
            "fill_missing",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "one_hot",
            OneHotEncoder(handle_unknown="ignore"),
        ),
    ]
)

In [50]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numbers", numeric_pipeline, numeric_features),
        ("categories", category_pipeline, category_features),
    ]
)

In [ ]:
# Latest training season becomes the validation set.
validation_labels = np.where(
    train_data["season"] == validation_year,
    0,
    -1,
)
time_validation = PredefinedSplit(validation_labels)

positive_count = int(y_train.sum())
negative_count = len(y_train) - positive_count
class_ratio = negative_count / positive_count

In [52]:
models = {
    "Random Forest": {
        "model": RandomForestClassifier(
            random_state=RANDOM_STATE,
            class_weight="balanced",
            n_jobs=-1,
        ),
        "parameters": {
            "model__n_estimators": [300, 500],
            "model__max_depth": [5, 8],
            "model__min_samples_leaf": [3, 6],
        },
    },
    "XGBoost": {
        "model": xgb.XGBClassifier(
            random_state=RANDOM_STATE,
            eval_metric="logloss",
            scale_pos_weight=class_ratio,
            n_jobs=-1,
        ),
        "parameters": {
            "model__n_estimators": [250, 400],
            "model__max_depth": [2, 3],
            "model__learning_rate": [0.03, 0.06],
            "model__subsample": [0.8],
            "model__colsample_bytree": [0.8],
            "model__reg_lambda": [3, 8],
        },
    },
}


In [53]:
trained_models = {}
comparison = []

for model_name, details in models.items():
    print("Training:", model_name)

    pipeline = Pipeline(
        steps=[
            ("prepare", preprocessor),
            ("model", details["model"]),
        ]
    )

    search = GridSearchCV(
        pipeline,
        details["parameters"],
        scoring="neg_log_loss",
        cv=time_validation,
        n_jobs=-1,
    )
    search.fit(X_train, y_train)

    trained_models[model_name] = search.best_estimator_
    comparison.append(
        {
            "model": model_name,
            "validation_log_loss": -search.best_score_,
            "best_parameters": search.best_params_,
        }
    )


Training: Random Forest
Training: XGBoost


In [54]:
model_comparison = pd.DataFrame(comparison).sort_values(
    "validation_log_loss"
).reset_index(drop=True)
display(model_comparison)

,model,validation_log_loss,best_parameters
0,Random Forest,0.141438,"{'model__max_depth': 8, 'model__min_samples_le..."
1,XGBoost,0.145145,"{'model__colsample_bytree': 0.8, 'model__learn..."


In [ ]:
best_model_name = model_comparison.loc[0, "model"]
best_model = trained_models[best_model_name]

In [56]:
raw_probability = best_model.predict_proba(X_test)[:, 1]

predictions = test_data[
    [
        "season",
        "round",
        "race_name",
        "driver",
        "team",
        "grid_position",
        "finish_position",
        "winner",
    ]
].copy()

predictions["race_id"] = (
    predictions["season"].astype(str)
    + "_"
    + predictions["round"].astype(str).str.zfill(2)
)
predictions["raw_probability"] = raw_probability


In [57]:
# Force probabilities within each race to add up to 100%.
race_total = predictions.groupby("race_id")[
    "raw_probability"
].transform("sum")
predictions["winner_probability"] = (
    predictions["raw_probability"] / race_total
)

test_loss = log_loss(
    y_test,
    predictions["winner_probability"],
    labels=[0, 1],
)
test_auc = roc_auc_score(
    y_test,
    predictions["winner_probability"],
)


In [ ]:
predicted_winners = (
    predictions.sort_values(
        ["race_id", "winner_probability"],
        ascending=[True, False],
    )
    .groupby("race_id")
    .first()
    .reset_index()
    .rename(columns={"driver": "predicted_winner"})
)

actual_winners = (
    predictions[predictions["winner"] == 1]
    [["race_id", "driver"]]
    .rename(columns={"driver": "actual_winner"})
)

In [59]:
race_summary = predicted_winners.merge(
    actual_winners,
    on="race_id",
)
race_summary["correct"] = (
    race_summary["predicted_winner"]
    == race_summary["actual_winner"]
)

race_accuracy = race_summary["correct"].mean()

In [60]:
print("Best model:", best_model_name)
print(f"Test log loss: {test_loss:.4f}")
print(f"Test ROC AUC: {test_auc:.4f}")
print(f"Race winner accuracy: {race_accuracy:.2%}")
print(
    "Different predicted winners:",
    race_summary["predicted_winner"].nunique(),
)


Best model: Random Forest
Test log loss: 0.1056
Test ROC AUC: 0.9747
Race winner accuracy: 60.00%
Different predicted winners: 2


In [61]:
display(
    race_summary[
        [
            "round",
            "race_name",
            "predicted_winner",
            "actual_winner",
            "winner_probability",
            "correct",
        ]
    ]
)


,round,race_name,predicted_winner,actual_winner,winner_probability,correct
0,1,Australian Grand Prix,NOR,NOR,0.315797,True
1,2,Chinese Grand Prix,NOR,PIA,0.189871,False
2,3,Japanese Grand Prix,NOR,VER,0.215655,False
3,4,Bahrain Grand Prix,PIA,PIA,0.400060,True
4,5,Saudi Arabian Grand Prix,PIA,PIA,0.282829,True


In [62]:
model_package = {
    "model": best_model,
    "model_name": best_model_name,
    "feature_columns": feature_columns,
    "numeric_features": numeric_features,
    "category_features": category_features,
    "test_year": TEST_YEAR,
    "test_log_loss": test_loss,
    "test_auc": test_auc,
    "race_accuracy": race_accuracy,
    "uses_driver_identity": False,
    "uses_team_identity": False,
    "data_sessions": [
        "FP1",
        "FP2",
        "FP3",
        "Qualifying",
        "Sprint Qualifying/Shootout",
        "Sprint",
    ],
}

joblib.dump(model_package, MODEL_FILE)
print("Saved:", MODEL_FILE.resolve())


Saved: /Users/santhoshkumarv/vs_code_projects/internship-harshith/projects/capstone_project/artifacts/f1_session_lap_winner_model.joblib


In [63]:
SAMPLE_ROUND = None

if SAMPLE_ROUND is None:
    sample_round = int(test_data["round"].max())
else:
    sample_round = SAMPLE_ROUND

sample = test_data[
    test_data["round"] == sample_round
].copy()

if sample.empty:
    raise ValueError("The selected round is unavailable.")

loaded_model = joblib.load(MODEL_FILE)["model"]
sample_raw = loaded_model.predict_proba(
    sample[feature_columns]
)[:, 1]
sample["winner_probability"] = sample_raw / sample_raw.sum()

sample_output = sample[
    [
        "race_name",
        "driver",
        "team",
        "grid_position",
        "winner_probability",
        "winner",
    ]
].sort_values("winner_probability", ascending=False)

predicted = sample_output.iloc[0]["driver"]
actual = sample_output[
    sample_output["winner"] == 1
].iloc[0]["driver"]


In [64]:
print("Race:", sample_output.iloc[0]["race_name"])
print("Predicted winner:", predicted)
print("Actual winner:", actual)
print("Correct:", predicted == actual)


Race: Saudi Arabian Grand Prix
Predicted winner: PIA
Actual winner: PIA
Correct: True


In [65]:
display(
    sample_output.head(10).style.format(
        {
            "grid_position": "{:.0f}",
            "winner_probability": "{:.2%}",
        }
    )
)

,race_name,driver,team,grid_position,winner_probability,winner
380,Saudi Arabian Grand Prix,PIA,McLaren,2,28.28%,1
383,Saudi Arabian Grand Prix,NOR,McLaren,10,21.88%,0
381,Saudi Arabian Grand Prix,VER,Red Bull,1,21.44%,0
384,Saudi Arabian Grand Prix,RUS,Mercedes,3,16.54%,0
382,Saudi Arabian Grand Prix,LEC,Ferrari,4,3.38%,0
399,Saudi Arabian Grand Prix,GAS,Alpine F1 Team,9,2.38%,0
385,Saudi Arabian Grand Prix,ANT,Mercedes,5,1.88%,0
388,Saudi Arabian Grand Prix,ALB,Williams,11,0.81%,0
398,Saudi Arabian Grand Prix,TSU,Red Bull,8,0.55%,0
389,Saudi Arabian Grand Prix,HAD,RB F1 Team,14,0.55%,0
